In [1]:
refusal_tokens = ['I', 'Sorry', 'cannot', "can't", 'unable', 'apologize']
harmful_tokens = ['Kill', 'hurt', 'stab', 'attack', 'injure', 'harm']
tl_model= HookedTransformer.from_pretrained('gpt2-medium')
def get_token_ids(token_list):
    token_ids=[tl_model.to_single_token(t) for t in token_list]
    return token_ids

def score_probs(probs, harmful_ids, refusal_ids):
    harmful_probs=probs[harmful_ids].sum.items()
    refusal_probs=probs[refusal_ids].sum.items()
    return harmful_probs
    
def run_full_analysis(prompt, max_new_tokens=20):
    refusal_ids = get_token_ids(refusal_tokens)
    harmful_ids = get_token_ids(harmful_tokens)
    
    current_ids = tl_model.to_tokens(prompt)
    
    all_layer_results = {i: {'delta_harmful': 0, 'delta_refusal': 0} 
                         for i in range(tl_model.cfg.n_layers)}
    all_harmful_probs=[]
    all_refusal_probs=[]
    for step in range(max_new_tokens):
        # 1. get baseline logits AND cache for this step
        logits, cache = tl_model.run_with_cache(current_ids)
        probs = torch.softmax(logits[0, -1, :], dim=-1)
        base_harmful, base_refusal = score_probs(probs, harmful_ids, refusal_ids)
        all_harmful_probs.append(base_harmful)
        all_refusal_probs.append(base_refusal)
        avg_refusal_probs=
        # 2. for each layer, mean ablate using THIS step's cache
        for layer_idx in range(tl_model.cfg.n_layers):
            def hook_fn(value, hook):
                # use cache from THIS step — dimensions always match
                value[:] = cache[hook.name].mean(dim=1, keepdim=True)
                return value
            
            abl_logits = tl_model.run_with_hooks(
                current_ids,
                fwd_hooks=[(f"blocks.{layer_idx}.hook_mlp_out", hook_fn)]
            )
            abl_probs = torch.softmax(abl_logits[0, -1, :], dim=-1)
            abl_harmful, abl_refusal = score_probs(abl_probs, harmful_ids, refusal_ids)
            
            all_layer_results[layer_idx]['delta_harmful'] += base_harmful - abl_harmful
            all_layer_results[layer_idx]['delta_refusal'] += abl_refusal - base_refusal
        
        # 3. greedy decode next token and append
        next_token = logits[0, -1, :].argmax().unsqueeze(0).unsqueeze(0)
        current_ids = torch.cat([current_ids, next_token], dim=1)
    
    # average over steps
    for layer_idx in all_layer_results:
        all_layer_results[layer_idx]['delta_harmful'] /= max_new_tokens
        all_layer_results[layer_idx]['delta_refusal'] /= max_new_tokens
        all_layer_results[layer_idx]['circuit_score'] = (
            all_layer_results[layer_idx]['delta_harmful'] + 
            all_layer_results[layer_idx]['delta_refusal']
        )
    
    return all_layer_results

NameError: name 'HookedTransformer' is not defined